In [1]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment(
    "CatBoost_30Trials-with-HPT"
)

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/18 00:31:52 INFO mlflow.tracking.fluent: Experiment with name 'CatBoost_30Trials-with-HPT' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/Soham/Documents/youtube comment '
 'analyzer/mlruns/232065981429596679'), creation_time=1779044512644, experiment_id='232065981429596679', last_update_time=1779044512644, lifecycle_stage='active', name='CatBoost_30Trials-with-HPT', tags={}, trace_location=None, workspace='default'>

In [3]:
# =========================================================
# IMPORTS
# =========================================================
import numpy as np
import pandas as pd
import optuna
import mlflow

from catboost import CatBoostClassifier

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.utils.class_weight import (
    compute_sample_weight
)

import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# RANDOM SEED
# =========================================================
np.random.seed(42)

# =========================================================
# LOAD DATASET
# =========================================================
df_original = pd.read_csv(
    "reddit_preprocessing.csv"
).dropna()

# =========================================================
# SAFE COPY
# =========================================================
df = df_original.copy()

# =========================================================
# MAP LABELS
# CatBoost prefers 0,1,2 labels
# =========================================================
df['category'] = df['category'].map({

    -1: 2,

    0: 0,

    1: 1
})

# =========================================================
# REMOVE NaN
# =========================================================
df = df.dropna(
    subset=['category']
)

# =========================================================
# VERIFY CLASSES
# =========================================================
print("=" * 60)

print("CLASSES:")

print(df['category'].unique())

print("\nCLASS COUNTS:\n")

print(df['category'].value_counts())

print("=" * 60)

# =========================================================
# FEATURES / TARGET
# =========================================================
X_text = df['clean_comment']

y = df['category']

# =========================================================
# TRAIN TEST SPLIT
# =========================================================
X_train_text, X_test_text, y_train, y_test = train_test_split(

    X_text,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

# =========================================================
# SAMPLE WEIGHTS
# =========================================================
sample_weights = compute_sample_weight(

    class_weight='balanced',

    y=y_train
)

# =========================================================
# TF-IDF SETTINGS
# =========================================================
ngram_range = (1, 3)

max_features = 5000

# =========================================================
# CROSS VALIDATION
# =========================================================
cv = StratifiedKFold(

    n_splits=3,

    shuffle=True,

    random_state=42
)

# =========================================================
# OPTUNA OBJECTIVE FUNCTION
# =========================================================
def objective_catboost(trial):

    params = {

        "iterations": trial.suggest_int(
            "iterations",
            100,
            400
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.20,
            log=True
        ),

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            10
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            5
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            5
        ),

        # =================================================
        # MULTICLASS SETTINGS
        # =================================================
        "loss_function": "MultiClass",

        "eval_metric": "TotalF1",

        "random_seed": 42,

        "verbose": 0,

        "thread_count": 4
    }

    # =====================================================
    # PIPELINE
    # =====================================================
    pipeline = Pipeline([

        (
            "tfidf",

            TfidfVectorizer(

                ngram_range=ngram_range,

                max_features=max_features,

                min_df=3,

                max_df=0.90,

                sublinear_tf=True
            )
        ),

        (
            "model",

            CatBoostClassifier(
                **params
            )
        )
    ])

    # =====================================================
    # CROSS VALIDATION
    # =====================================================
    scores = cross_val_score(

        pipeline,

        X_train_text,

        y_train,

        cv=cv,

        scoring='f1_macro',

        n_jobs=1
    )

    return np.mean(scores)

# =========================================================
# RUN OPTUNA
# =========================================================
study = optuna.create_study(
    direction="maximize"
)

study.optimize(

    objective_catboost,

    n_trials=30
)

# =========================================================
# BEST PARAMETERS
# =========================================================
print("=" * 60)

print("BEST PARAMETERS")

print(study.best_params)

print("=" * 60)

# =========================================================
# FINAL TF-IDF
# =========================================================
vectorizer = TfidfVectorizer(

    ngram_range=ngram_range,

    max_features=max_features,

    min_df=3,

    max_df=0.90,

    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(
    X_train_text
)

X_test_vec = vectorizer.transform(
    X_test_text
)

# =========================================================
# FINAL MODEL
# =========================================================
best_model = CatBoostClassifier(

    **study.best_params,

    loss_function="MultiClass",

    eval_metric="TotalF1",

    random_seed=42,

    verbose=0,

    thread_count=4
)

# =========================================================
# TRAIN FINAL MODEL
# =========================================================
best_model.fit(

    X_train_vec,

    y_train,

    sample_weight=sample_weights
)

# =========================================================
# PREDICTIONS
# =========================================================
y_pred = best_model.predict(
    X_test_vec
)

# =========================================================
# METRICS
# =========================================================
accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(

    y_test,

    y_pred,

    average='macro'
)

weighted_f1 = f1_score(

    y_test,

    y_pred,

    average='weighted'
)

# =========================================================
# RESULTS
# =========================================================
print(f"\nAccuracy    : {accuracy:.4f}")

print(f"Macro F1    : {macro_f1:.4f}")

print(f"Weighted F1 : {weighted_f1:.4f}")

# =========================================================
# CLASSIFICATION REPORT
# =========================================================
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# =========================================================
# CONFUSION MATRIX
# =========================================================
conf_matrix = confusion_matrix(
    y_test,
    y_pred
)

fig, ax = plt.subplots(figsize=(8,6))

sns.heatmap(

    conf_matrix,

    annot=True,

    fmt='d',

    cmap='Blues',

    xticklabels=[0,1,2],

    yticklabels=[0,1,2],

    ax=ax
)

ax.set_xlabel("Predicted")

ax.set_ylabel("Actual")

ax.set_title(
    "CatBoost 3-Class Confusion Matrix"
)

# =========================================================
# MLFLOW LOGGING
# =========================================================
with mlflow.start_run(
    run_name="CatBoost_30Trials"
):

    mlflow.set_tag(
        "model",
        "CatBoost"
    )

    mlflow.set_tag(
        "vectorizer",
        "TF-IDF"
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "macro_f1",
        macro_f1
    )

    mlflow.log_metric(
        "weighted_f1",
        weighted_f1
    )

    mlflow.log_figure(
        fig,
        "catboost_confusion_matrix.png"
    )

    mlflow.sklearn.log_model(

        sk_model=best_model,

        name="catboost_model"
    )

plt.show()

plt.close(fig)

[I 2026-05-18 00:40:59,653] A new study created in memory with name: no-name-dd77b7d4-6dc2-4c83-81f0-369996e728aa


CLASSES:
[1 2 0]

CLASS COUNTS:

category
1    15770
0    12644
2     8248
Name: count, dtype: int64


[I 2026-05-18 00:56:33,485] Trial 0 finished with value: 0.6030909577835004 and parameters: {'iterations': 361, 'learning_rate': 0.02617483965700121, 'depth': 7, 'l2_leaf_reg': 5.278428002796255, 'random_strength': 1.0833435936104858, 'bagging_temperature': 4.478167159404957}. Best is trial 0 with value: 0.6030909577835004.
[I 2026-05-18 01:01:28,545] Trial 1 finished with value: 0.5745053891092587 and parameters: {'iterations': 106, 'learning_rate': 0.03851502971390475, 'depth': 7, 'l2_leaf_reg': 3.3110636034192673, 'random_strength': 2.8412396787513696, 'bagging_temperature': 3.52211935780336}. Best is trial 0 with value: 0.6030909577835004.
[W 2026-05-18 04:03:04,317] Trial 2 failed with parameters: {'iterations': 336, 'learning_rate': 0.025331575681286335, 'depth': 10, 'l2_leaf_reg': 6.061405596372325, 'random_strength': 4.546773837817888, 'bagging_temperature': 3.1982770163517045} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\

KeyboardInterrupt: 